# NammaSpace — Cloud Reconstruction (Colab GPU)

Runs the **heavy** reconstruction on a free Colab GPU (T4 ~15 GB), then exports a
compressed `.ksplat` + `scene.json` you drop straight into `viewer/public/scenes/`.
This sidesteps the 4 GB local VRAM limit and lets us train uncapped (more frames,
more splats, higher res, 30k steps) for a sharper, more complete room.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

> Uses nerfstudio's `splatfacto` (CUDA — reliable on Colab). Reuses this repo's
> `build_scene.py` (COLMAP → scene.json) and `convert-ksplat.mjs` (compression).
> Note: nerfstudio's install cell is the one most likely to need a version tweak;
> if it errors, read the message — usually a torch/cuda pin.

In [ ]:
# 1. Confirm the GPU (want ~15 GB T4 / L4 / A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. System tools + this repo (for build_scene.py & convert-ksplat.mjs)
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null && echo 'colmap + ffmpeg ok'
%cd /content
![ -d Nammaspace ] || git clone -q https://github.com/ShashwatK27/Nammaspace.git
print('repo ready')

In [ ]:
# 3. Install nerfstudio (splatfacto trainer). Takes several minutes.
#    If this errors on torch/cuda, pin torch to Colab's version first, then rerun.
!pip -q install nerfstudio 2>&1 | tail -3
!ns-train --help > /dev/null 2>&1 && echo 'nerfstudio ok' || echo 'nerfstudio install needs a fix — read the pip output above'

In [ ]:
# 4. Upload your room video (or mount Drive and set VIDEO=path)
from google.colab import files
up = files.upload()
VIDEO = list(up.keys())[0]
print('using video:', VIDEO)

In [ ]:
# 5. Frames + COLMAP poses (nerfstudio runs ffmpeg + COLMAP). ~300 frames.
!ns-process-data video --data "$VIDEO" --output-dir /content/proc --num-frames-target 300
# Convert the COLMAP model to TXT so our build_scene.py can read it
import glob, os
sparse = sorted(glob.glob('/content/proc/**/sparse/0', recursive=True) + glob.glob('/content/proc/colmap/sparse/0'))
SPARSE = sparse[0]; print('colmap model:', SPARSE)
!colmap model_converter --input_path "$SPARSE" --output_path "$SPARSE" --output_type TXT

In [ ]:
# 6. Train splatfacto (uncapped for a big GPU). ~15-40 min depending on GPU.
!ns-train splatfacto --data /content/proc \
  --max-num-iterations 30000 \
  --viewer.quit-on-train-completion True \
  --output-dir /content/out

In [ ]:
# 7. Export the trained splat to .ply
import glob
CFG = sorted(glob.glob('/content/out/**/config.yml', recursive=True))[-1]
print('config:', CFG)
!ns-export gaussian-splat --load-config "$CFG" --output-dir /content/export
PLY = sorted(glob.glob('/content/export/*.ply'))[-1]; print('ply:', PLY)

In [ ]:
# 8. Compress .ply -> .ksplat (10x) using this repo's converter
%cd /content/Nammaspace/viewer
!npm -q install @mkkellogg/gaussian-splats-3d 2>&1 | tail -1
!node convert-ksplat.mjs "$PLY" /content/scene.ksplat 1 0

In [ ]:
# 9. Build the scene.json bundle (bounds/spawn/up) from the COLMAP model
%cd /content/Nammaspace
!python reconstruction/pipeline/build_scene.py \
  --model "$SPARSE" --out /content/bundle \
  --splat /content/scene.ksplat \
  --config reconstruction/config/pipeline.json
!ls -la /content/bundle /content/bundle/splat

In [ ]:
# 10. Download the bundle (scene.json + scene.ksplat)
import shutil
from google.colab import files
shutil.make_archive('/content/scene_bundle', 'zip', '/content/bundle')
files.download('/content/scene_bundle.zip')

## Drop it into the viewer
1. Unzip → you get `scene.json` + `splat/scene.ksplat`.
2. Put them in `viewer/public/scenes/<scene-id>/` (e.g. `venue`).
3. Point the viewer at it: set `SCENE_URL = '/scenes/<scene-id>/scene.json'` in `viewer/src/main.js`.
4. Commit + push → Vercel redeploys the live link.

### Optional SOTA upgrade (later)
Swap step 6 for **2D Gaussian Splatting** (`hbb1/2d-gaussian-splatting`, CUDA — builds on Colab)
for flatter walls + an extractable mesh (useful for Round 2 nav/AR occlusion), or
**MASt3R-SfM** in place of COLMAP for faster/denser poses. Both are heavier to set up;
get this reliable pipeline working first.